# 03 — ProofFrame Recheck and Rule Overlays

Four capabilities introduced in routemap Batches 4 and 5a/b/c:

1. **ProofFrame Rechecker** (`recheck_proof_frame`) — given a `SupportArtifact` (the per-atom proof evidence Q1 Check returns) and an `EvaluationOverlay`, recheck per-atom verdicts and report a frame-level status: `still_valid` / `invalidated` / `unknown`.
2. **Rule Disable** (`check_rule_disable_action`) — temporarily drop one body atom from a rule and observe the variant rows.
3. **Rule Literal Replace** (`check_rule_literal_replace_action`) — swap a constant literal in a body atom (e.g. `region == 'us'` → `region == 'eu'`).
4. **Rule Add Condition** (`check_rule_add_condition_action`) — append one filter atom (no new variable binder) to a rule body branch.

All three rule overlays are **single-action** MVPs that return both `variant_rows` (the new rows the rewritten rule would produce) and a nested `proof_frame: ProofFrameRecheckResult` describing how the original frame is affected.

**Prerequisites:** [02_overlay_why_not_frontier.ipynb](02_overlay_why_not_frontier.ipynb). **Next:** [04_round_persistence_diff.ipynb](04_round_persistence_diff.ipynb).

## Setup

Same fixture shape as the previous chapters; this time we include `dave` (age 17, us) so the Add Condition demo (`lt($age, 20)`) has exactly one matching row.

In [ ]:
from __future__ import annotations

import sys
from dataclasses import dataclass
from pathlib import Path

_repo_root = Path.cwd()
if not (_repo_root / 'src').exists() and (_repo_root.parent / 'src').exists():
    _repo_root = _repo_root.parent
_src_dir = _repo_root / 'src'
if str(_src_dir) not in sys.path:
    sys.path.insert(0, str(_src_dir))

from kernel.application import (
    build_fact_value_override,
    build_schema_index,
    check_derivation_binding,
    check_rule_add_condition_action,
    check_rule_disable_action,
    check_rule_literal_replace_action,
    entity_info,
    field_predicate,
    recheck_proof_frame,
    resolve_selector,
)
from kernel.application.protocol import (
    CheckRequest,
    CompiledDerivationPlan,
    CompiledHeadCall,
    EntitySelector,
    EvaluationOverlay,
    FieldPath,
    ProofFrameRecheckRequest,
    RuleAddConditionAction,
    RuleAddConditionRequest,
    RuleAddedAtom,
    RuleDisableAction,
    RuleDisableRequest,
    RuleLiteralPath,
    RuleLiteralReplaceAction,
    RuleLiteralReplaceRequest,
)
from kernel.core.evidence.write_protocol import set_field
from kernel.core.rules.rule_ir import RuleSpec
from kernel.core.store import Store
from kernel.core.store._support import NonFactStep, PredWitness, SupportArtifact
from kernel.sdk import Entity, Field, Identity, compile_schema_from_classes


class Person(Entity):
    name: str = Identity(primary_key=True)
    age: int = Field(cardinality='single')
    region: str = Field(cardinality='single')


@dataclass(frozen=True)
class SeededPerson:
    e_ref: str
    age: int
    region: str
    exists_pred_id: str
    age_pred_id: str
    region_pred_id: str
    exists_asrt_id: str
    age_asrt_id: str
    region_asrt_id: str


def seed_person(store, index, *, name, age, region):
    info = entity_info(index, 'Person')
    ref = resolve_selector(
        EntitySelector(entity_type='Person', identity={'name': name}),
        index=index,
    )
    encoded = ref.encoded_ref or ''
    exists_asrt_id = set_field(store.ledger, info.exists_predicate_id, encoded, [])
    set_field(store.ledger, info.identity_predicates['name'].pred_id, encoded, [('string', name)])
    age_pred = field_predicate(index, 'Person', 'age').pred_id
    region_pred = field_predicate(index, 'Person', 'region').pred_id
    age_asrt_id = set_field(store.ledger, age_pred, encoded, [('int', age)])
    region_asrt_id = set_field(store.ledger, region_pred, encoded, [('string', region)])
    return SeededPerson(
        e_ref=encoded, age=age, region=region,
        exists_pred_id=info.exists_predicate_id,
        age_pred_id=age_pred, region_pred_id=region_pred,
        exists_asrt_id=exists_asrt_id,
        age_asrt_id=age_asrt_id, region_asrt_id=region_asrt_id,
    )


def _binding(*items):
    return tuple(sorted(items, key=lambda item: item[0]))


schema_ir = compile_schema_from_classes([Person])
index = build_schema_index(schema_ir)
store = Store(schema_ir)
people = {
    'alice': seed_person(store, index, name='alice', age=25, region='us'),
    'bob':   seed_person(store, index, name='bob',   age=30, region='eu'),
    'carol': seed_person(store, index, name='carol', age=28, region='us'),
    'dave':  seed_person(store, index, name='dave',  age=17, region='us'),
}
person_info = entity_info(index, 'Person')
plan = CompiledDerivationPlan(
    derivation_id='round-story-person-snapshot',
    version='1.0',
    body_ir=[
        ('pred', person_info.exists_predicate_id, ['$p']),
        ('pred', field_predicate(index, 'Person', 'age').pred_id,    ['$p', '$age']),
        ('pred', field_predicate(index, 'Person', 'region').pred_id, ['$p', '$region']),
    ],
    heads=(CompiledHeadCall(
        target_pred_id=person_info.exists_predicate_id,
        head_var_names=('$p', '$age', '$region'),
    ),),
)
print(f'Seeded {len(people)} people including dave (age 17) for the add-condition phase.')

## 1. Obtain a `SupportArtifact` from Q1 Check

The ProofFrame Rechecker operates on the `SupportArtifact` returned by Q1 Check (`evidence_envelope.engine_payload`). The artifact carries per-atom witnesses (`PredWitness`) keyed under the plan body, plus any non-fact steps. We run a passing Check against Alice to obtain the artifact.

In [ ]:
alice = people['alice']
binding_alice = _binding(('$p', alice.e_ref), ('$age', 25), ('$region', 'us'))
check_result = check_derivation_binding(
    CheckRequest(plan=plan, binding=binding_alice, engine='native'),
    store=store,
)

assert check_result.status == 'passed'
support_from_check = check_result.evidence_envelope.engine_payload
assert isinstance(support_from_check, SupportArtifact)

print(f'Check status                : {check_result.status}')
print(f'support kind                : {support_from_check.kind}')
print(f'support binding_items       : {support_from_check.binding_items}')
print(f'pred_witness count          : {len(support_from_check.pred_witnesses)}')
for witness in support_from_check.pred_witnesses:
    print(f'  pred_atom_key={witness.pred_atom_key}  asrt_ids={witness.asrt_ids}')

## 2. ProofFrame Rechecker (Batch 4)

`recheck_proof_frame(ProofFrameRecheckRequest(support_artifact, overlay), store=...)` answers: *given the per-atom proof Alice's success rested on, does that proof still hold under this overlay?*

We make two recheck calls against the same support artifact:

- **Baseline** with an empty `EvaluationOverlay()`. Expected: `still_valid`.
- **Overlay** with the same Alice age 25→30 override from chapter 2. Expected: `invalidated`, with the age atom marked as `invalidated` in `atom_verdicts`.

In [ ]:
baseline_result = recheck_proof_frame(
    ProofFrameRecheckRequest(
        support_artifact=support_from_check,
        overlay=EvaluationOverlay(),
    ),
    store=store,
)
assert baseline_result.status == 'still_valid'
print(f'baseline status     : {baseline_result.status}')
print(f'baseline atom_verdicts:')
for verdict in baseline_result.atom_verdicts:
    print(f'  atom {verdict.atom_key}: verdict={verdict.verdict}')

In [ ]:
age_override_overlay = EvaluationOverlay(
    fact_actions=(
        build_fact_value_override(
            store, index,
            e_ref=alice.e_ref,
            field=FieldPath(entity_type='Person', field_name='age'),
            new_value=30,
            note='demo overlay: Alice turns 30',
        ),
    ),
)

overlay_result = recheck_proof_frame(
    ProofFrameRecheckRequest(
        support_artifact=support_from_check,
        overlay=age_override_overlay,
    ),
    store=store,
)

assert overlay_result.status == 'invalidated'
assert any(v.verdict == 'invalidated' for v in overlay_result.atom_verdicts)

print(f'overlay status      : {overlay_result.status}')
print(f'overlay atom_verdicts:')
for verdict in overlay_result.atom_verdicts:
    print(
        f'  atom {verdict.atom_key}: verdict={verdict.verdict} '
        f'affected_action_indices={verdict.affected_action_indices}'
    )

## 3. Build a `RuleSpec` and a manual `SupportArtifact` for the rule overlays

The three rule-overlay capabilities (Disable / Literal Replace / Add Condition) operate on a `RuleSpec` (the rule under test) and a `SupportArtifact` describing the *baseline* row whose proof we want to perturb.

For demo determinism we hand-construct the support artifact from Alice's seeded `asrt_id`s — atom keys follow the convention `b{branch}.a{atom_index}:{pred_id}` for predicate atoms and `b{branch}.a{atom_index}:{kind}` for non-fact steps. (In a real workflow this artifact would also come from a prior Check call; constructing it inline here keeps every value visible.)

The rule under test is `name(?p) ∧ age(?p, ?age) ∧ region(?p, ?region) ∧ region == 'us'` — three predicate atoms followed by an `eq` non-fact step.

In [ ]:
rule_spec = RuleSpec(
    rule_id='person.eligible',
    version='1.0',
    select_vars=['$p'],
    where=[
        ('pred', person_info.exists_predicate_id,                              ['$p']),
        ('pred', field_predicate(index, 'Person', 'age').pred_id,              ['$p', '$age']),
        ('pred', field_predicate(index, 'Person', 'region').pred_id,           ['$p', '$region']),
        ('eq', '$region', 'us'),
    ],
)

rule_support = SupportArtifact(
    kind='native_binding_v1',
    root_result_kind='row',
    binding_items=(('$p', alice.e_ref),),
    pred_witnesses=(
        PredWitness(pred_atom_key=f'b0.a0:{alice.exists_pred_id}', asrt_ids=(alice.exists_asrt_id,)),
        PredWitness(pred_atom_key=f'b0.a1:{alice.age_pred_id}',    asrt_ids=(alice.age_asrt_id,)),
        PredWitness(pred_atom_key=f'b0.a2:{alice.region_pred_id}', asrt_ids=(alice.region_asrt_id,)),
    ),
    non_fact_steps=(
        NonFactStep(step_key='b0.a3:eq', kind='eq', status='satisfied'),
    ),
)

print(f'rule_id      : {rule_spec.rule_id}')
print(f'where atoms  : {len(rule_spec.where)} (3 pred + 1 eq)')
print(f'support binding_items: {rule_support.binding_items}')

## 4. Rule Disable (Batch 5a) — drop one body atom

`check_rule_disable_action(RuleDisableRequest(rule_spec, support_artifact, overlay))` evaluates the rule body with one designated atom dropped. The overlay's `rule_actions` field carries one `RuleDisableAction` identifying `(branch_index, atom_index)`.

Below we disable atom index 3 (the `('eq', '$region', 'us')` non-fact step), widening the matching universe from "us-only" to "any region". `variant_rows` reports the new rows; `proof_frame.status` reports whether Alice's original frame survives (it doesn't — the rule semantics changed).

In [ ]:
rule_disable_action = RuleDisableAction(
    rule_id='person.eligible',
    version='1.0',
    branch_index=0,
    atom_index=3,
)

disable_result = check_rule_disable_action(
    RuleDisableRequest(
        rule_spec=rule_spec,
        support_artifact=rule_support,
        overlay=EvaluationOverlay(rule_actions=(rule_disable_action,)),
    ),
    store=store,
)

assert disable_result.status == 'completed'
assert disable_result.variant_rows
assert disable_result.proof_frame.status == 'invalidated'

print(f'status              : {disable_result.status}')
print(f'variant_rows count  : {len(disable_result.variant_rows)}')
for row in disable_result.variant_rows:
    print(f'  variant: {row}')
print(f'proof_frame.status  : {disable_result.proof_frame.status}')

## 5. Rule Literal Replace (Batch 5b) — swap a constant literal

`check_rule_literal_replace_action(...)` swaps a constant literal inside a body atom. The action carries a `RuleLiteralPath(kind='rhs')` (the right-hand side of the equality), the `old_literal`, and the `new_literal`.

Below we change the rule's `region == 'us'` to `region == 'eu'`. The new universe should match exactly Bob (the only seeded person whose region is 'eu').

In [ ]:
bob = people['bob']
rule_literal_replace_action = RuleLiteralReplaceAction(
    rule_id='person.eligible',
    version='1.0',
    branch_index=0,
    atom_index=3,
    literal_path=RuleLiteralPath(kind='rhs'),
    old_literal='us',
    new_literal='eu',
)

replace_result = check_rule_literal_replace_action(
    RuleLiteralReplaceRequest(
        rule_spec=rule_spec,
        support_artifact=rule_support,
        overlay=EvaluationOverlay(rule_actions=(rule_literal_replace_action,)),
    ),
    store=store,
)

assert replace_result.status == 'completed'
assert replace_result.variant_rows == ((('$p', bob.e_ref),),)
assert replace_result.proof_frame.status == 'invalidated'

print(f'status              : {replace_result.status}')
print(f'variant_rows        : {replace_result.variant_rows}')
print(f'proof_frame.status  : {replace_result.proof_frame.status}')

## 6. Rule Add Condition (Batch 5c) — append a filter atom

`check_rule_add_condition_action(...)` appends one extra filter atom to a rule body branch. The added atom is wrapped in a `RuleAddedAtom(...)` carrying the same `('kind', ...args)` shape used in the where-body.

Batch 5c ships the *filter-only* slice — added atoms must not introduce new variable binders. Below we append `lt($age, 20)`, narrowing the matching universe to Dave (the only seeded person under 20).

In [ ]:
dave = people['dave']
rule_add_action = RuleAddConditionAction(
    rule_id='person.eligible',
    version='1.0',
    branch_index=0,
    added_atom=RuleAddedAtom(('lt', '$age', 20)),
)

add_result = check_rule_add_condition_action(
    RuleAddConditionRequest(
        rule_spec=rule_spec,
        support_artifact=rule_support,
        overlay=EvaluationOverlay(rule_actions=(rule_add_action,)),
    ),
    store=store,
)

assert add_result.status == 'completed'
assert add_result.variant_rows == ((('$p', dave.e_ref),),)
assert add_result.proof_frame.status == 'invalidated'

print(f'status              : {add_result.status}')
print(f'variant_rows        : {add_result.variant_rows}')
print(f'proof_frame.status  : {add_result.proof_frame.status}')

## Wrap-up

| Capability | API | Module |
|---|---|---|
| ProofFrame Rechecker (Batch 4) | `recheck_proof_frame(ProofFrameRecheckRequest)`               | `kernel.application.proofframe_runtime` |
| Rule Disable (Batch 5a)        | `check_rule_disable_action(RuleDisableRequest)`               | `kernel.application.rule_disable_runtime` |
| Rule Literal Replace (Batch 5b)| `check_rule_literal_replace_action(RuleLiteralReplaceRequest)`| `kernel.application.rule_literal_replace_runtime` |
| Rule Add Condition (Batch 5c)  | `check_rule_add_condition_action(RuleAddConditionRequest)`    | `kernel.application.rule_add_condition_runtime` |

**Composition note:** All three rule-overlay capabilities return a `proof_frame: ProofFrameRecheckResult` alongside `variant_rows`. The proof frame describes how the *original* support is affected by the rule rewrite — a single call answers both "what new rows would the variant rule produce?" and "does the original derivation still stand?"

**Public-surface boundary (Batch 8):** All four runtimes are advanced-importable from `kernel.application`. v0.1 ships **no** SDK shells or service routes for them.

**Next:** [04_round_persistence_diff.ipynb](04_round_persistence_diff.ipynb) — Batch 6 durable round events (`kernel.audit.round_events`) + Batch 7 ProofFrame diff (`AuditQuery.diff_proof_frames`).